# Exploration of the RUEG Corpus
## Gathering MetaData Files

In [33]:
import glob
files = glob.glob('../Data/Input/RUEG-1.0_corpora/core/en/exb/RUEG/EN/BILINGUAL/*.meta', recursive = True)
bi_meta_files = []
for f in files:
    file = open(f)
    s = file.read()
    point = f.split("BILINGUAL/",1)[1].strip('.meta')
    bi_meta_files.append((point, s))
    file.close()

files = glob.glob('../Data/Input/RUEG-1.0_corpora/core/en/exb/RUEG/EN/MONOLINGUAL/*.meta', recursive = True)
mono_meta_files= []
for f in files:
    file = open(f)
    s = file.read()
    point = f.split("MONOLINGUAL/",1)[1].strip('.meta')
    mono_meta_files.append((point, s))
    file.close()

In [28]:
print('Bilingual Files:', len(bi_meta_files))
print('Monolingual Files:', len(mono_meta_files))

Bilingual Files: 892
Monolingual Files: 256


In [29]:
bi_meta_files[0]

'USbi66FT_fsE'

In [12]:
files = glob.glob('../Data/Input/RUEG-1.0_corpora/core/en/conll/RUEG/EN/BILINGUAL/*.conllu', recursive = True)
bi_texts = []
bi_files = []
for file in files:
    f = open(file)
    s = f.read()
    f1 = file.split("BILINGUAL/",1)[1].strip('.conllu')
    bi_texts.append((f1, s))
    f.close()
    bi_files.append(file)

In [15]:

files = glob.glob('../Data/Input/RUEG-1.0_corpora/core/en/conll/RUEG/EN/MONOLINGUAL/*.conllu', recursive = True)
mono_texts = []
mono_files = []
for file in files:
    f = open(file)
    s = f.read()
    f1 = file.split("MONOLINGUAL/",1)[1].strip('.conllu')
    mono_texts.append((f1, s))
    f.close()
    mono_files.append(file)

In [16]:
print("Bilingual text files:", len(bi_texts))
print("Monolingual text files:", len(mono_texts))

Bilingual text files: 892
Monolingual text files: 256


In [ ]:
bi_texts[0]

## files may be .conll, but they are still parsable in python
## they are just tab seperated- I will only be extracting POS information for now
## but there's a lot more information in the conll format that can be used

('USbi94FG_iwE',
 '1\ta\ta\tDET\tAT0\t_\t2\tdet\t_\t_\n2\tguy\tguy\tNOUN\tNN1\t_\t4\tnsubj\t_\t_\n3\twas\tbe\tAUX\tVBD\t_\t4\tcop\t_\t_\n4\twalking\twalk\tVERB\tVVG\t_\t0\troot\t_\t_\n5\twith\twith\tADP\tPRP\t_\t7\tcase\t_\t_\n6\this\the\tDET\tDPS\t_\t7\tdet\t_\t_\n7\twife\twife\tNOUN\tNN1\t_\t4\tobl\t_\t_\n8\tand\tand\tCCONJ\tCJC\t_\t9\tcc\t_\t_\n9\tkid\tkid\tNOUN\tNN1\t_\t4\tconj\t_\t_\n\n1\tlooked\tlook\tVERB\tVVD\t_\t0\troot\t_\t_\n2\tlike\tlike\tSCONJ\tCJS\t_\t5\tmark\t_\t_\n3\tthey\tthey\tPRON\tPNP\t_\t5\tnsubj\t_\t_\n4\twere\tbe\tAUX\tVBD\t_\t5\taux\t_\t_\n5\tgoing\tgo\tVERB\tVVG\t_\t1\tadvcl\t_\t_\n6\tto\tto\tADP\tPRP\t_\t8\tcase\t_\t_\n7\tthe\tthe\tDET\tAT0\t_\t8\tdet\t_\t_\n8\tpark\tpark\tNOUN\tNN1\t_\t5\tobl\t_\t_\n\n1\tand\tand\tCCONJ\tCJC\t_\t4\tcc\t_\t_\n2\tthey\tthey\tPRON\tPNP\t_\t4\tnsubj\t_\t_\n3\twere\tbe\tAUX\tVBD\t_\t4\tcop\t_\t_\n4\tcrossing\tcross\tVERB\tVVG\t_\t0\troot\t_\t_\n5\tthe\tthe\tDET\tAT0\t_\t6\tdet\t_\t_\n6\tstreet\tstreet\tNOUN\tNN1\t_\t4\tobj\t_\t_\n

## Parsing ConllU
The parsing code here comes from this [past project](https://github.com/Data-Science-for-Linguists-2025/DEU-ENG-Mono-and-Billingual-Speakers/blob/main/LoadingRUEGData.ipynb)

In [ ]:
import stanza
from stanza.utils.conll import CoNLL
from stanza.models.common.doc import Document
import nltk
from itertools import chain


def convert_conllu_to_pos(input_path, pos_list):
    """
    This code takes in an input path and a part of speech list.
    It parses the file and returns tuples of WORD and POS.

    Args:
        input_path: the path to the conllU data file
        pos_list: an empty list

    Returns:
        pos_list: list of lists of sentences containing each word and POS
            Example: ('and', 'CCONJ)
    """
    with open(input_path, 'r', encoding='utf-8') as file:
        lines = file.readlines()
    
    pos_text = ""
    sentence = list([tuple()])
    
    for line in lines:
        line = line.strip()
        if line and not line.startswith("#"):
            columns = line.split("\t")
            if len(columns) > 3:
                word_text = columns[1]  # Token
                upos = columns[3]  # Universal POS Tag

                extension = tuple([word_text, upos])
                sentence.append(extension)
        else:
            if sentence:
                pos_text = sentence
                sentence = []
    
    pos_list.append(pos_text)

In [ ]:
bi_pos = []
flat_bi_pos = []
bigram_bi_pos = []
trigram_bi_pos = []
for files in bi_files:
    convert_conllu_to_pos(files, bi_pos)
for x in bi_pos:
    bigram_bi_pos.extend(list(nltk.bigrams(x)))
    trigram_bi_pos.extend(list(nltk.ngrams(x, 3)))

flat_bi_pos.extend(chain(*bi_pos))

print(bi_pos[0])    # list of sentences
print(flat_bi_pos[:10])     # raw list of words and POS (flattened sentences)
print(bigram_bi_pos[:5])    # list of bigram tuples
print(trigram_bi_pos[:3])   # list of trigram tuples

[('and', 'CCONJ'), ('they', 'PRON'), ('called', 'VERB'), ('911', 'PROPN'), ('to', 'PART'), ('tell', 'VERB'), ('them', 'PRON'), ('what', 'DET'), ('happened', 'VERB')]
[('and', 'CCONJ'), ('they', 'PRON'), ('called', 'VERB'), ('911', 'PROPN'), ('to', 'PART'), ('tell', 'VERB'), ('them', 'PRON'), ('what', 'DET'), ('happened', 'VERB'), ('and', 'CCONJ')]
[(('and', 'CCONJ'), ('they', 'PRON')), (('they', 'PRON'), ('called', 'VERB')), (('called', 'VERB'), ('911', 'PROPN')), (('911', 'PROPN'), ('to', 'PART')), (('to', 'PART'), ('tell', 'VERB'))]
[(('and', 'CCONJ'), ('they', 'PRON'), ('called', 'VERB')), (('they', 'PRON'), ('called', 'VERB'), ('911', 'PROPN')), (('called', 'VERB'), ('911', 'PROPN'), ('to', 'PART'))]


In [23]:
mono_pos = []
flat_mono_pos = []
bigram_mono_pos = []
trigram_mono_pos = []
for files in mono_files:
    convert_conllu_to_pos(files, mono_pos)
for x in mono_pos:
    bigram_mono_pos.extend(list(nltk.bigrams(x)))
    trigram_mono_pos.extend(list(nltk.ngrams(x, 3)))

flat_mono_pos.extend(chain(*mono_pos))

print(mono_pos[0])  # list of sentences
print(flat_mono_pos[:10])   # raw list of words and POS
print(bigram_mono_pos[:5])  # list of bigram tuples
print(trigram_mono_pos[:3]) # list of trigram tuples

[('I', 'PRON'), ('could', 'AUX'), ('not', 'PART'), ('tell', 'VERB'), ('how', 'ADV'), ('bad', 'ADJ'), ('his', 'DET'), ('car', 'NOUN'), ('was', 'AUX')]
[('I', 'PRON'), ('could', 'AUX'), ('not', 'PART'), ('tell', 'VERB'), ('how', 'ADV'), ('bad', 'ADJ'), ('his', 'DET'), ('car', 'NOUN'), ('was', 'AUX'), ('Good', 'ADJ')]
[(('I', 'PRON'), ('could', 'AUX')), (('could', 'AUX'), ('not', 'PART')), (('not', 'PART'), ('tell', 'VERB')), (('tell', 'VERB'), ('how', 'ADV')), (('how', 'ADV'), ('bad', 'ADJ'))]
[(('I', 'PRON'), ('could', 'AUX'), ('not', 'PART')), (('could', 'AUX'), ('not', 'PART'), ('tell', 'VERB')), (('not', 'PART'), ('tell', 'VERB'), ('how', 'ADV'))]


In [24]:
## length + average sentence length
print(len(flat_bi_pos))
print(len(flat_mono_pos))
print(len(flat_bi_pos)/len(bi_pos))
print(len(flat_mono_pos)/len(mono_pos))

8574
2530
9.612107623318385
9.8828125


Right off the bat, we can see that there are a lot more bilingual files than monolingual files. We can also see that the monolingual sentences are slightly longer, although not by a lot. 